In [1]:
%pip install astropy wmm-calculator

  Using cached wmm_calculator-1.4.3-py3-none-any.whl.metadata (11 kB)
  Using cached geomaglib-1.2.3-py3-none-any.whl.metadata (6.8 kB)
Using cached wmm_calculator-1.4.3-py3-none-any.whl (14 kB)
Using cached geomaglib-1.2.3-py3-none-any.whl (3.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [wmm-calculator]
Note: you may need to restart the kernel to use updated packages.


In [6]:
"""
Satellite + tether inertia models (about bus COM) for 3 configurations.

ORDER:
  1) Coiled/stowed tether inside bus as an annular cylinder (spool) + stowed point mass (ALWAYS included)
     - end-mass position can be: known input, or random-in-spool (seeded)

  2) Fully extended tether aligned with -z_B (point mass at end; tether taut at nominal length)

  3) BENT TETHER CONFIG (orbit + current + WMM2025 + astropy; stiff integrator)
     - Earth-centered inertial (origin at Earth center), meters
     - circular orbit model for bus COM: altitude + inclination (+ optional RAAN, nu0)
     - geomagnetic field from WMM via wmm-calculator package (wmm_calc)
     - astropy used for ECI<->ECEF (GCRS<->ITRS) and geodetic lat/lon/height
     - tether modeled as lumped-mass chain (nodes) with axial spring-damper, tension-only option
     - endpoint mass is separate (not part of uniform node masses)
     - final output: inertia tensor in inertial axes about the bus COM at settle time (or t_final)

PATCHES INCLUDED:
  A) WMM scalar extraction fixed for wmm-calculator outputs that are 0-d or (1,) arrays
  B) IERS warn-and-continue (auto-download enabled for best accuracy when internet is available)
  C) solve_ivp speed + "not stuck" behavior:
        - jac_sparsity provided (prevents dense finite-diff Jacobian)
        - chunked integration with progress prints
        - rhs NaN/inf tripwire

DEPENDENCIES:
  - numpy
  - scipy (solve_ivp)
  - astropy
  - wmm-calculator  (pip install wmm-calculator)
"""

from __future__ import annotations
from dataclasses import dataclass
from typing import Tuple, Optional, Callable, Dict, Any
import numpy as np
def smooth_max0(x: float, eps: float) -> float:
    """Smooth approximation of max(x, 0). eps sets the transition width."""
    eps = float(eps)
    if eps <= 0.0:
        return max(float(x), 0.0)
    x = float(x)
    return 0.5 * (x + (x*x + eps*eps) ** 0.5)

import time

# ---------- SciPy (for config 3) ----------
try:
    from scipy.integrate import solve_ivp
    from scipy.optimize import root_scalar
    from scipy.sparse import lil_matrix
    from scipy.io import savemat
except Exception:
    solve_ivp = None
    lil_matrix = None

# ---------- Astropy (for config 3 + WMM sampling) ----------
ASTROPY_OK = True
try:
    import astropy.units as u
    from astropy.time import Time
    from astropy.utils import iers
    from astropy.coordinates import (
        SkyCoord,
        GCRS,
        ITRS,
        CartesianRepresentation,
    )

    # === PATCH: best-accuracy IERS settings (warn-and-continue) ===
    iers.conf.auto_download = True
    iers.conf.iers_degraded_accuracy = "warn"  # warn, then continue

    def ensure_iers_warmup() -> None:
        """
        Optional warm-up: tries to load IERS-A now so you don't get surprised later.
        If it fails (no internet / blocked), it warns and continues.
        """
        try:
            _ = iers.IERS_Auto.open()
        except Exception as e:
            print(f"  [IERS warn] Could not pre-load IERS-A ({e}). Continuing with degraded accuracy if needed.", flush=True)

except Exception:
    ASTROPY_OK = False

# ---------- WMM calculator ----------
WMM_OK = True
try:
    from wmm import wmm_calc
except Exception:
    WMM_OK = False


# ============================================================
# Basic linear algebra helpers
# ============================================================

def hat(v: np.ndarray) -> np.ndarray:
    vx, vy, vz = v
    return np.array([[0.0, -vz,  vy],
                     [vz,  0.0, -vx],
                     [-vy, vx,  0.0]], dtype=float)

def unit(v: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    n = float(np.linalg.norm(v))
    if n < tol:
        raise ValueError("Cannot normalize a near-zero vector.")
    return v / n


# ============================================================
# Inertia building blocks
# ============================================================

def parallel_axis_shift(m: float, r: np.ndarray) -> np.ndarray:
    r = r.reshape(3)
    return m * ((np.dot(r, r) * np.eye(3)) - np.outer(r, r))

def point_mass_inertia_about_origin(m: float, r: np.ndarray) -> np.ndarray:
    return parallel_axis_shift(m, r)

def box_inertia_about_com(m: float, a: float, b: float, c: float) -> np.ndarray:
    Ixx = (m / 12.0) * (b**2 + c**2)
    Iyy = (m / 12.0) * (a**2 + c**2)
    Izz = (m / 12.0) * (a**2 + b**2)
    return np.diag([Ixx, Iyy, Izz])

def annular_cylinder_inertia_about_com(
    m: float,
    r_i: float,
    r_o: float,
    h: float,
    axis_s_hat_B: np.ndarray
) -> np.ndarray:
    s = unit(axis_s_hat_B)
    I_parallel = 0.5 * m * (r_i**2 + r_o**2)
    I_perp = (m / 12.0) * (3.0 * (r_i**2 + r_o**2) + h**2)
    P = np.outer(s, s)
    return I_perp * (np.eye(3) - P) + I_parallel * P

def slender_rod_inertia_about_com(m: float, L: float, axis_e_hat: np.ndarray) -> np.ndarray:
    e = unit(axis_e_hat)
    return (m * L**2 / 12.0) * (np.eye(3) - np.outer(e, e))

def inertia_body_to_inertial(I_B: np.ndarray, R_IB: np.ndarray) -> np.ndarray:
    return R_IB @ I_B @ R_IB.T


# ============================================================
# Random stowed end-mass position inside spool
# ============================================================

def rotation_matrix_from_a_to_b(a: np.ndarray, b: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    a_hat = unit(a)
    b_hat = unit(b)
    v = np.cross(a_hat, b_hat)
    s = float(np.linalg.norm(v))
    c = float(np.dot(a_hat, b_hat))

    if s < tol:
        if c > 0:
            return np.eye(3)
        tmp = np.array([1.0, 0.0, 0.0])
        if abs(np.dot(tmp, a_hat)) > 0.9:
            tmp = np.array([0.0, 1.0, 0.0])
        axis = unit(np.cross(a_hat, tmp))
        K = hat(axis)
        return np.eye(3) + 2.0 * (K @ K)

    K = hat(v / s)
    return np.eye(3) + K * s + (K @ K) * (1.0 - c)

def sample_point_in_annular_cylinder(
    r_i: float, r_o: float, h: float,
    axis_s_B: np.ndarray,
    com_B: np.ndarray,
    rng: np.random.Generator
) -> np.ndarray:
    u_rand = rng.random()
    r = np.sqrt(r_i**2 + u_rand * (r_o**2 - r_i**2))
    phi = 2.0 * np.pi * rng.random()
    z = (rng.random() - 0.5) * h

    p_local = np.array([r * np.cos(phi), r * np.sin(phi), z], dtype=float)
    R = rotation_matrix_from_a_to_b(np.array([0.0, 0.0, 1.0]), axis_s_B)
    return (R @ p_local) + com_B


# ============================================================
# Data containers
# ============================================================

@dataclass
class BusParams:
    mass: float
    a: float
    b: float
    c: float

@dataclass
class SpoolParams:
    mass: float
    r_i: float
    r_o: float
    h: float
    axis_s_B: np.ndarray
    com_B: np.ndarray

@dataclass
class TetherParams:
    length: float
    mass: float
    attach_B: np.ndarray

@dataclass
class PointMassParams:
    mass: float


# ============================================================
# CONFIG 1: coiled/spool + stowed point mass
# ============================================================

def moi_config_1_coiled(bus: BusParams, spool: SpoolParams, m_end: float, r_end_B: np.ndarray) -> np.ndarray:
    I_bus = box_inertia_about_com(bus.mass, bus.a, bus.b, bus.c)
    I_spool_com = annular_cylinder_inertia_about_com(
        spool.mass, spool.r_i, spool.r_o, spool.h, spool.axis_s_B
    )
    I_spool = I_spool_com + parallel_axis_shift(spool.mass, spool.com_B)
    I_end = point_mass_inertia_about_origin(m_end, r_end_B)
    return I_bus + I_spool + I_end


# ============================================================
# CONFIG 2: deployed tether along -z_B
# ============================================================

def moi_config_2_extended_on_minus_z(
    bus: BusParams,
    tether: TetherParams,
    end_mass: PointMassParams
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    I_bus = box_inertia_about_com(bus.mass, bus.a, bus.b, bus.c)

    L = tether.length
    z_hat = np.array([0.0, 0.0, 1.0])
    r_a_B = tether.attach_B.reshape(3)

    # along -z_B:
    r_p_B0 = r_a_B - L * z_hat

    I_pm = point_mass_inertia_about_origin(end_mass.mass, r_p_B0)

    e_hat = -z_hat
    r_c_B = r_a_B - 0.5 * L * z_hat
    I_rod_com = slender_rod_inertia_about_com(tether.mass, L, e_hat)
    I_rod = I_rod_com + parallel_axis_shift(tether.mass, r_c_B)

    return I_bus + I_pm + I_rod, r_p_B0, r_a_B


# ============================================================
# CONFIG 3 support: orbit + attitude + WMM field
# ============================================================

def Rx(phi: float) -> np.ndarray:
    c, s = np.cos(phi), np.sin(phi)
    return np.array([[1, 0, 0],
                     [0, c, -s],
                     [0, s,  c]], dtype=float)

def Rz(psi: float) -> np.ndarray:
    c, s = np.cos(psi), np.sin(psi)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]], dtype=float)

def Ry(theta: float) -> np.ndarray:
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[ c, 0, s],
                     [ 0, 1, 0],
                     [-s, 0, c]], dtype=float)

def euler_zyx_to_R(roll_x: float, pitch_y: float, yaw_z: float) -> np.ndarray:
    return Rz(yaw_z) @ Ry(pitch_y) @ Rx(roll_x)

def rad(deg_: float) -> float:
    return deg_ * np.pi / 180.0

def deg(rad_: float) -> float:
    return rad_ * 180.0 / np.pi

def circular_orbit_eci_rv(
    t: float,
    alt_m: float,
    inc_deg: float,
    raan_deg: float,
    nu0_deg: float,
    mu: float = 3.986004418e14,
    Re: float = 6378137.0
) -> Tuple[np.ndarray, np.ndarray, float]:
    a = Re + alt_m
    n = np.sqrt(mu / a**3)
    nu = rad(nu0_deg) + n * float(t)

    inc = rad(inc_deg)
    raan = rad(raan_deg)

    R = Rz(raan) @ Rx(inc) @ Rz(nu)

    r_pf = np.array([a, 0.0, 0.0], dtype=float)
    v_pf = np.array([0.0, a * n, 0.0], dtype=float)

    r = R @ r_pf
    v = R @ v_pf
    T = (2.0 * np.pi / n)
    return r, v, float(T)

def make_R_IB_callable(
    att_mode: str,
    roll_deg: float = 0.0,
    pitch_deg: float = 0.0,
    yaw_deg: float = 0.0
) -> Callable[[float], np.ndarray]:
    if att_mode.lower() == "i":
        return lambda t: np.eye(3)
    if att_mode.lower() == "e":
        R0 = euler_zyx_to_R(rad(roll_deg), rad(pitch_deg), rad(yaw_deg))
        return lambda t: R0
    raise ValueError("att_mode must be 'i' or 'e'.")


# ---------- WMM helper (patched scalar extraction) ----------

def make_wmm_callable_wmm_calculator() -> Callable[[Time, float, float, float], np.ndarray]:
    if not WMM_OK:
        raise ImportError(
            "Could not import 'wmm_calc'. Install with:\n"
            "  pip install wmm-calculator\n"
            "and verify: from wmm import wmm_calc"
        )

    model = wmm_calc()

    def _as_float(v) -> float:
        a = np.asarray(v)
        return float(a.item()) if a.size == 1 else float(a.ravel()[0])

    def B_ned_T(time_astropy: Time, lat_deg: float, lon_deg: float, alt_m: float) -> np.ndarray:
        y = int(time_astropy.utc.datetime.year)
        m = int(time_astropy.utc.datetime.month)
        d = int(time_astropy.utc.datetime.day)

        model.setup_time(y, m, d)
        model.setup_env(float(lat_deg), float(lon_deg), float(alt_m), unit="m")
        out = model.get_all()  # nT in NED: x=north, y=east, z=down

        Bn = _as_float(out["x"])
        Be = _as_float(out["y"])
        Bd = _as_float(out["z"])
        return 1e-9 * np.array([Bn, Be, Bd], dtype=float)

    return B_ned_T

def ned_to_ecef(lat_rad: float, lon_rad: float) -> np.ndarray:
    sl = np.sin(lat_rad); cl = np.cos(lat_rad)
    so = np.sin(lon_rad); co = np.cos(lon_rad)

    n = np.array([-sl*co, -sl*so,  cl], dtype=float)
    e = np.array([-so,     co,     0.0], dtype=float)
    uvec = np.array([ cl*co,  cl*so,  sl], dtype=float)
    d = -uvec
    return np.column_stack([n, e, d])

def itrs_vec_to_gcrs_vec(B_ecef_T: np.ndarray, obstime: Time) -> np.ndarray:
    if not ASTROPY_OK:
        raise ImportError("astropy is required for ITRS<->GCRS transforms.")
    c_itrs = SkyCoord(
        CartesianRepresentation(B_ecef_T[0]*u.T, B_ecef_T[1]*u.T, B_ecef_T[2]*u.T),
        frame=ITRS(obstime=obstime)
    )
    c_gcrs = c_itrs.transform_to(GCRS(obstime=obstime))
    return c_gcrs.cartesian.xyz.to_value(u.T)

def eci_pos_to_geodetic_latlonalt(r_eci_m: np.ndarray, obstime: Time) -> Tuple[float, float, float]:
    if not ASTROPY_OK:
        raise ImportError("astropy is required for coordinate transforms.")
    sc_gcrs = SkyCoord(
        CartesianRepresentation(r_eci_m[0]*u.m, r_eci_m[1]*u.m, r_eci_m[2]*u.m),
        frame=GCRS(obstime=obstime)
    )
    sc_itrs = sc_gcrs.transform_to(ITRS(obstime=obstime))
    loc = sc_itrs.earth_location
    lat = loc.lat.to_value(u.deg)
    lon = loc.lon.to_value(u.deg)
    alt = loc.height.to_value(u.m)
    return float(lat), float(lon), float(alt)

def make_B_eci_callable(
    epoch_utc: str,
    alt_m: float,
    inc_deg: float,
    raan_deg: float,
    nu0_deg: float,
    n_samples: int = 200
) -> Tuple[Callable[[float], np.ndarray], Dict[str, Any]]:
    if solve_ivp is None:
        raise ImportError("scipy is required for config 3 (solve_ivp).")
    if not ASTROPY_OK:
        raise ImportError("astropy is required for config 3.")
    if not WMM_OK:
        raise ImportError("wmm-calculator is required for config 3 (pip install wmm-calculator).")

    epoch = Time(epoch_utc, scale="utc")

    # orbit period
    _, _, T = circular_orbit_eci_rv(0.0, alt_m, inc_deg, raan_deg, nu0_deg)
    t_grid = np.linspace(0.0, float(T), int(n_samples), dtype=float)

    B_ned = make_wmm_callable_wmm_calculator()
    B_grid = np.zeros((len(t_grid), 3), dtype=float)

    t0 = time.perf_counter()
    for i, t in enumerate(t_grid):
        if i == 0 or i == len(t_grid)-1 or (i % max(1, len(t_grid)//8) == 0):
            print(f"    WMM precompute {i}/{len(t_grid)}", flush=True)

        obstime = epoch + float(t) * u.s
        r_com, _, _ = circular_orbit_eci_rv(float(t), alt_m, inc_deg, raan_deg, nu0_deg)

        lat_deg, lon_deg, alt_here_m = eci_pos_to_geodetic_latlonalt(r_com, obstime)

        B_ned_T = B_ned(obstime, lat_deg, lon_deg, alt_here_m)
        C = ned_to_ecef(np.deg2rad(lat_deg), np.deg2rad(lon_deg))
        B_ecef_T = C @ B_ned_T
        B_eci_T = itrs_vec_to_gcrs_vec(B_ecef_T, obstime)

        B_grid[i, :] = B_eci_T

    t1 = time.perf_counter()
    print(f"  B-field precompute finished in {t1 - t0:.2f} s", flush=True)
    print(f"     precomputed_to_s = {float(T):.3f}", flush=True)
    print(f"     B_grid_shape     = {B_grid.shape}", flush=True)
    print(f"     sample B(0) [T]  = {B_grid[0]}", flush=True)

    def B_eci_of_t(t: float) -> np.ndarray:
        t = float(t)
        if t <= t_grid[0]:
            return B_grid[0].copy()
        if t >= t_grid[-1]:
            return B_grid[-1].copy()
        j = int(np.searchsorted(t_grid, t) - 1)
        j = max(0, min(j, len(t_grid)-2))
        t0_, t1_ = t_grid[j], t_grid[j+1]
        w = (t - t0_) / (t1_ - t0_)
        return (1.0 - w) * B_grid[j] + w * B_grid[j+1]

    meta = {
        "epoch": epoch,
        "t_grid": t_grid,
        "B_grid_T": B_grid,
        "period_s": float(T),
    }
    return B_eci_of_t, meta


# ============================================================
# CONFIG 3: bent tether dynamics + inertia at settle time
# ============================================================

@dataclass
class BentTetherSimParams:
    epoch_utc: str
    alt_m: float
    inc_deg: float
    raan_deg: float
    nu0_deg: float

    # attitude
    att_mode: str                 # 'i' or 'e'
    roll_deg: float = 0.0
    pitch_deg: float = 0.0
    yaw_deg: float = 0.0

    # electrodynamic tether current profile (bare tether + hollow cathode)

    # --- current-profile parameters ---
    V_bias_V: float = 50.0             # applied bias / supply [V] (set 0 for passive)
    V_cath_V: float = 10.0             # hollow cathode voltage drop [V]
    plasma_density_m3: float = 5.0e11  # N_inf [1/m^3]
    tether_diameter_m: float = 0.001   # cylindrical tether diameter [m]
    tether_sigma_Spm: float = 3.5e7    # conductivity [S/m]
    bare_fraction: float = 1.0         # fraction of tether that is bare/collecting [0..1]
    scale_with_Em: bool = True         # scale current with motional E-field Em(t)
    current_turnon_tau_s: float = 0.0  # optional smooth turn-on [s] (0 => instant)

    # discretization
    N_seg: int = 40
    EA_N: float = 5e5
    c_Ns_m: float = 800.0
    tension_only: bool = True

    # integration
    t_final: float = 1200.0
    n_output: int = 1200

    # settle detection
    v_tol: float = 1e-3
    hold_time: float = 30.0

    # initial tether direction in inertial
    init_dir_mode: str = "z"         # 'z' => use -z_I, 'v' => user vector
    init_dir_I: Optional[np.ndarray] = None

    # WMM sampling along orbit
    B_samples: int = 200

# ------------------------------------------------------------
# Bare tether current profile (dimensionless ODE + shooting)
# ------------------------------------------------------------

def _tether_ode_collect(x: float, y: np.ndarray) -> Tuple[float, float]:
    """Dimensionless ODE from the current-profile solver notebook."""
    phi, current = float(y[0]), float(y[1])
    dphi_dx = current + 1.0
    if phi >= 0.0:
        di_dx = 0.75 * (phi ** 0.5)
    else:
        di_dx = 0.0
    return dphi_dx, di_dx


def _auto_bracket(f: Callable[[float], float], start: float = -50.0, step: float = 5.0, max_val: float = 200.0) -> Tuple[float, float]:
    a = float(start)
    fa = float(f(a))
    b = a + float(step)
    while b <= float(max_val):
        fb = float(f(b))
        if np.sign(fa) != np.sign(fb):
            return a, b
        a, fa = b, fb
        b += float(step)
    raise RuntimeError("No valid bracket found for shooting. Try different parameters.")


def compute_dimless_current_profile(
    x_max: float,
    x_B: float,
    phi_target: float,
    n_points: int = 2000
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Solve for dimensionless potential phi(x) and current i(x) on 0..x_max.
    On 0..x_B tether is bare/collecting; on x_B..x_max current is saturated (di/dx=0).
    Boundary: i(0)=0 and phi(x_max)=phi_target (implemented via shooting).
    """
    x_max = float(x_max)
    x_B = float(x_B)
    phi_target = float(phi_target)

    if x_max <= 0.0:
        x = np.array([0.0], dtype=float)
        return x, np.zeros_like(x), np.zeros_like(x)

    x_B = max(0.0, min(x_B, x_max))

    def rhs(x, y):
        dphi, di = _tether_ode_collect(x, y)
        return [dphi, di]

    def shooting_residual(phi0_guess: float) -> float:
        sol = solve_ivp(
            rhs,
            (0.0, x_B if x_B > 0.0 else 0.0),
            [float(phi0_guess), 0.0],
            dense_output=True,
            max_step=max(x_B / 2000.0, 1e-6) if x_B > 0.0 else 1e-6
        )
        if x_B > 0.0:
            phi_B, i_B = sol.sol(x_B)
            phi_B = float(phi_B); i_B = float(i_B)
        else:
            # no collecting region
            phi_B, i_B = float(phi0_guess), 0.0

        # insulated region: di/dx=0, dphi/dx=i+1
        phi_pred = phi_B + (i_B + 1.0) * (x_max - x_B)
        return float(phi_pred - phi_target)

    # find phi0 via bracketing
    a_br, b_br = _auto_bracket(shooting_residual, start=-50.0, step=5.0, max_val=200.0)
    phi0 = root_scalar(shooting_residual, bracket=[a_br, b_br], method="brentq", xtol=1e-6).root

    # integrate once with the converged phi0
    sol = solve_ivp(
        rhs,
        (0.0, x_B if x_B > 0.0 else 0.0),
        [float(phi0), 0.0],
        dense_output=True,
        max_step=max(x_B / 2000.0, 1e-6) if x_B > 0.0 else 1e-6
    )

    x = np.linspace(0.0, x_max, int(n_points), dtype=float)
    phi = np.zeros_like(x)
    i_prof = np.zeros_like(x)

    if x_B > 0.0:
        maskA = x <= x_B
        phi[maskA], i_prof[maskA] = sol.sol(x[maskA])
        phi_B, i_B = sol.sol(x_B)
        phi_B = float(phi_B); i_B = float(i_B)
    else:
        # no collecting region
        i_B = 0.0
        phi_B = float(phi0)

    maskB = x > x_B
    i_prof[maskB] = i_B
    phi[maskB] = phi_B + (i_B + 1.0) * (x[maskB] - x_B)

    return x, phi, i_prof


def compute_bare_tether_current_profile_midpoints(
    L_total_m: float,
    seg_mid_s_m: np.ndarray,
    Em_ref_Vpm: float,
    V_bias_V: float,
    V_cath_V: float,
    plasma_density_m3: float,
    tether_diameter_m: float,
    tether_sigma_Spm: float,
    bare_fraction: float = 1.0,
    n_profile_pts: int = 2000
) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Returns base currents [A] at segment midpoints for a cylindrical bare tether with a hollow cathode.
    The shape is computed at Em_ref (positive scalar), and can be scaled later ~ Em(t)/Em_ref.
    """
    L_total_m = float(L_total_m)
    Em_ref_Vpm = float(abs(Em_ref_Vpm))
    if Em_ref_Vpm < 1e-12 or L_total_m <= 0.0:
        return np.zeros_like(seg_mid_s_m, dtype=float), {"Em_ref_Vpm": Em_ref_Vpm}

    # geometry
    d = float(tether_diameter_m)
    A_c = np.pi * (d * 0.5) ** 2
    p_b = np.pi * d

    # constants
    e = 1.602176634e-19
    m_e = 9.10938356e-31
    N_inf = float(plasma_density_m3)
    sigma = float(tether_sigma_Spm)

    # characteristic scales (from the solver notebook)
    L_star = (2.0 * A_c / p_b) ** (2.0 / 3.0) * ((9.0 * np.pi**2 * m_e * sigma**2 * Em_ref_Vpm) / (128.0 * e**3 * N_inf**2)) ** (1.0 / 3.0)
    V_star = Em_ref_Vpm * L_star
    I_star = sigma * Em_ref_Vpm * A_c

    # dimensionless parameters
    eps_bc = float(V_bias_V) / V_star
    v_c = float(V_cath_V) / V_star
    phi_target = eps_bc - v_c

    x_max = L_total_m / L_star
    L_bare = max(0.0, min(float(bare_fraction), 1.0)) * L_total_m
    x_B = L_bare / L_star

    # solve dimless profile
    x, phi, i_prof = compute_dimless_current_profile(x_max=x_max, x_B=x_B, phi_target=phi_target, n_points=int(n_profile_pts))

    # evaluate at segment midpoints
    x_mid = np.asarray(seg_mid_s_m, dtype=float) / L_star
    i_mid = np.interp(x_mid, x, i_prof, left=i_prof[0], right=i_prof[-1])
    I_mid_A = i_mid * I_star

    meta = {
        "Em_ref_Vpm": Em_ref_Vpm,
        "L_star_m": float(L_star),
        "V_star_V": float(V_star),
        "I_star_A": float(I_star),
        "x_max": float(x_max),
        "x_B": float(x_B),
        "phi_target": float(phi_target),
    }
    return I_mid_A, meta


def turn_on_multiplier(t: float, tau_s: float) -> float:
    t = float(t)
    tau_s = float(tau_s)
    if tau_s <= 0.0:
        return 1.0
    return float(1.0 - np.exp(-t / tau_s))



def tether_chain_dynamics_rhs(
    t: float,
    y: np.ndarray,
    bus: BusParams,
    tether: TetherParams,
    end_mass: PointMassParams,
    sim: BentTetherSimParams,
    R_IB_of_t: Callable[[float], np.ndarray],
    B_eci_of_t: Callable[[float], np.ndarray],
    I_seg_base_A: Optional[np.ndarray] = None,
    Em_ref_Vpm: float = 0.0,
    u_current_I: Optional[np.ndarray] = None,
    mu: float = 3.986004418e14,
    Re: float = 6378137.0
) -> np.ndarray:
    """
    RELATIVE-STATE tether chain dynamics about the bus COM, expressed in inertial axes.

    State y packs:
      - r_rel[i] : particle position relative to bus COM (meters)
      - v_rel[i] : particle velocity relative to bus COM (m/s)

    This avoids ill-conditioning from integrating ~6.9e6 m absolute ECI positions and prevents
    "nodes near Earth's center" explosions caused by frame/scale mismatches.

    Includes:
      - differential gravity (gravity gradient): a_abs - a_com
      - axial spring-damper per segment
      - optional smooth tension-only clamp
      - Lorentz force per segment, distributed to endpoints
    """
    N_seg = int(sim.N_seg)
    if N_seg < 2:
        raise ValueError("Need N_seg >= 2.")
    N_internal = N_seg - 1
    N_particles = N_internal + 1

    r_rel = y[0:3*N_particles].reshape((N_particles, 3))
    v_rel = y[3*N_particles:6*N_particles].reshape((N_particles, 3))

    # bus COM orbit
    r_com, v_com, _ = circular_orbit_eci_rv(t, sim.alt_m, sim.inc_deg, sim.raan_deg, sim.nu0_deg, mu=mu, Re=Re)
    rcom_norm = float(np.linalg.norm(r_com))
    a_com = (-mu * r_com / (rcom_norm**3)) if rcom_norm > 1.0 else np.zeros(3)

    # attachment point relative to COM in inertial axes (attitude may be constant)
    R_IB = R_IB_of_t(t)
    r_attach_rel = (R_IB @ tether.attach_B.reshape(3))
    v_attach_rel = np.zeros(3, dtype=float)  # assume zero body-rate (attitude fixed)

    # absolute particle states (for gravity only)
    r_abs = r_rel + r_com
    v_abs = v_rel + v_com

    # rest length + stiffness
    L = float(tether.length)
    l0 = L / float(N_seg)
    k = float(sim.EA_N) / l0
    c = float(sim.c_Ns_m)

    # masses (OPTIONALLY scaled for dynamics; inertia uses true masses elsewhere)
    m_node = float(tether.mass) / float(N_internal) if N_internal > 0 else 0.0
    m_end = float(end_mass.mass)
    masses = np.full((N_particles,), m_node, dtype=float)
    masses[-1] = m_end
    masses *= float(getattr(sim, "mass_scale", 1.0))

    # forcing
    B = B_eci_of_t(t)

    # current along tether segments (segment-midpoint currents)
    if I_seg_base_A is None:
        Iseg = np.zeros((N_seg,), dtype=float)
    else:
        if bool(getattr(sim, 'scale_with_Em', True)) and (u_current_I is not None) and float(Em_ref_Vpm) > 1e-12:
            Em_t = float(np.dot(np.cross(v_com, B), u_current_I.reshape(3)))
            scale = Em_t / float(Em_ref_Vpm)
        else:
            scale = 1.0
        scale *= turn_on_multiplier(t, float(getattr(sim, 'current_turnon_tau_s', 0.0)))
        Iseg = np.asarray(I_seg_base_A, dtype=float) * scale


    # accelerations in RELATIVE coordinates: dv_rel/dt = a_abs - a_com
    a_rel = np.zeros_like(r_rel)

    # gravity gradient
    for i in range(N_particles):
        ri = r_abs[i]
        rnorm = float(np.linalg.norm(ri))
        if rnorm > 1.0:
            a_rel[i] += (-mu * ri / (rnorm**3)) - a_com

    # internal segment forces (use relative positions/velocities)
    r_nodes = np.vstack([r_attach_rel.reshape(1, 3), r_rel])
    v_nodes = np.vstack([v_attach_rel.reshape(1, 3), v_rel])

    # smooth clamp settings
    smooth_on = bool(getattr(sim, "smooth_tension_clamp", True))
    eps_frac = float(getattr(sim, "smooth_eps_frac", 1e-4))
    eps_len = eps_frac * l0
    eps_force = eps_frac * float(sim.EA_N)

    # iterate segments between node j-1 and j (j=1..N_particles)
    for j in range(1, N_particles + 1):
        r0 = r_nodes[j - 1]
        r1 = r_nodes[j]
        v0 = v_nodes[j - 1]
        v1 = v_nodes[j]

        dl = r1 - r0
        dist = float(np.linalg.norm(dl))
        if dist < 1e-12:
            continue
        e = dl / dist

        ext_raw = dist - l0
        if sim.tension_only:
            ext = smooth_max0(ext_raw, eps_len) if smooth_on else max(ext_raw, 0.0)
        else:
            ext = ext_raw

        rel_v_ax = float(np.dot((v1 - v0), e))
        f_raw = k * ext + c * rel_v_ax
        if sim.tension_only:
            f_mag = smooth_max0(f_raw, eps_force) if smooth_on else max(f_raw, 0.0)
        else:
            f_mag = f_raw

        # spring/damper force on node j (toward node j-1)
        f_vec = -f_mag * e

        # apply to particle at node j (index j-1)
        a_rel[j - 1] += f_vec / masses[j - 1]
        # equal/opposite on previous particle endpoint if it exists (node j-1 >= 1)
        if j - 1 >= 1:
            a_rel[j - 2] -= f_vec / masses[j - 2]

        # Lorentz force per segment, distributed to endpoints
        Ij = float(Iseg[j - 1])
        if Ij != 0.0:
            fL_seg = Ij * np.cross(dl, B)
            a_rel[j - 1] += 0.5 * fL_seg / masses[j - 1]
            if j - 1 >= 1:
                a_rel[j - 2] += 0.5 * fL_seg / masses[j - 2]

    dydt = np.zeros_like(y)
    dydt[0:3*N_particles] = v_rel.reshape(-1)
    dydt[3*N_particles:6*N_particles] = a_rel.reshape(-1)
    return dydt


def compute_inertia_from_particles_about_bus_com_in_inertial(
    r_particles_I: np.ndarray,
    m_particles: np.ndarray,
    r_com_I: np.ndarray
) -> np.ndarray:
    I = np.zeros((3, 3), dtype=float)
    for ri, mi in zip(r_particles_I, m_particles):
        rrel = ri - r_com_I
        I += mi * ((np.dot(rrel, rrel) * np.eye(3)) - np.outer(rrel, rrel))
    return I


# ---------- PATCH: Jacobian sparsity builder ----------
def build_jac_sparsity(Np: int):
    """
    Conservative sparsity pattern for y = [r(3Np), v(3Np)].

    dr_i depends on v_i
    dv_i depends on:
      - r_i, v_i
      - r_{i-1}, v_{i-1} (local coupling)
    """
    if lil_matrix is None:
        return None

    n = 6 * Np
    J = lil_matrix((n, n), dtype=int)

    # dr/dt w.r.t v
    for i in range(Np):
        J[3*i:3*i+3, 3*Np + 3*i:3*Np + 3*i+3] = 1

    # dv/dt w.r.t r,v (self + previous)
    for i in range(Np):
        row = 3*Np + 3*i

        # self
        J[row:row+3, 3*i:3*i+3] = 1
        J[row:row+3, 3*Np + 3*i:3*Np + 3*i+3] = 1

        # previous
        if i > 0:
            J[row:row+3, 3*(i-1):3*(i-1)+3] = 1
            J[row:row+3, 3*Np + 3*(i-1):3*Np + 3*(i-1)+3] = 1

    return J.tocsr()


def run_bent_tether_sim(
    bus: BusParams,
    tether: TetherParams,
    end_mass: PointMassParams,
    sim: BentTetherSimParams
) -> Dict[str, Any]:
    """
    Chunked stiff integration for the bent-tether case, patched for stability:
      - Integrates RELATIVE state about bus COM (meters-scale)
      - Auto mass scaling for "shape" convergence (reduces fastest-mode stiffness)
      - Vector tolerances tuned for relative coordinates
      - Optional early-stop settle detection
    """
    if solve_ivp is None:
        raise ImportError("scipy is required: pip install scipy")
    if not ASTROPY_OK:
        raise ImportError("astropy is required: pip install astropy")
    if not WMM_OK:
        raise ImportError("wmm-calculator is required: pip install wmm-calculator")

    ensure_iers_warmup()

    # --- defaults (safe even if BentTetherSimParams doesn't declare these) ---
    if not hasattr(sim, "use_relative_state"): sim.use_relative_state = True
    if not hasattr(sim, "early_stop"): sim.early_stop = True
    if not hasattr(sim, "T_target"): sim.T_target = 0.05
    if not hasattr(sim, "method"): sim.method = "BDF"
    if not hasattr(sim, "chunk_s"): sim.chunk_s = 60.0
    if not hasattr(sim, "atol_pos_m"): sim.atol_pos_m = 1e-2
    if not hasattr(sim, "atol_vel_mps"): sim.atol_vel_mps = 1e-5
    if not hasattr(sim, "mass_scale"): sim.mass_scale = 1.0
    if not hasattr(sim, "smooth_tension_clamp"): sim.smooth_tension_clamp = True
    if not hasattr(sim, "smooth_eps_frac"): sim.smooth_eps_frac = 1e-4
    if not hasattr(sim, "rtol"): sim.rtol = 1e-5

    # orbit + B field precompute
    print("\n  Precomputing WMM B-field along orbit at BUS COM (this is the heavy part)...", flush=True)
    B_eci_of_t, Bmeta = make_B_eci_callable(
        epoch_utc=sim.epoch_utc,
        alt_m=sim.alt_m,
        inc_deg=sim.inc_deg,
        raan_deg=sim.raan_deg,
        nu0_deg=sim.nu0_deg,
        n_samples=int(sim.B_samples)
    )
    R_IB_of_t = make_R_IB_callable(sim.att_mode, sim.roll_deg, sim.pitch_deg, sim.yaw_deg)

    # sizes
    N_seg = int(sim.N_seg)
    N_internal = N_seg - 1
    N_particles = N_internal + 1

    # rest length, stiffness diagnostic, auto mass scaling
    L = float(tether.length)
    l0 = L / float(N_seg)
    k = float(sim.EA_N) / l0
    m_node = float(tether.mass) / float(N_internal) if N_internal > 0 else 0.0
    T_fast = 2.0*np.pi*np.sqrt(max(m_node, 1e-12) / max(k, 1e-12))
    T_target = float(sim.T_target)
    if T_fast < T_target:
        ms_need = (T_target / max(T_fast, 1e-12))**2
        sim.mass_scale = max(float(sim.mass_scale), ms_need)

    print(f"  stiffness: N_seg={N_seg}, L={L:g} m, l0={l0:.3g} m, k={k:.3g} N/m, m_node={m_node:.3g} kg, "
          f"T_fast≈{T_fast:.3e} s, mass_scale={float(sim.mass_scale):.3g}", flush=True)

    # initial bus COM and attachment offset in inertial axes (RELATIVE to COM)
    r_com0, v_com0, _ = circular_orbit_eci_rv(0.0, sim.alt_m, sim.inc_deg, sim.raan_deg, sim.nu0_deg)
    R0 = R_IB_of_t(0.0)
    r_attach_rel0 = (R0 @ tether.attach_B.reshape(3))

    # initial direction in inertial axes
    if sim.init_dir_mode.lower() == "z":
        e0 = np.array([0.0, 0.0, -1.0], dtype=float)
    elif sim.init_dir_mode.lower() == "v":
        if sim.init_dir_I is None:
            raise ValueError("init_dir_I must be provided if init_dir_mode='v'")
        e0 = unit(sim.init_dir_I.reshape(3))
    else:
        raise ValueError("init_dir_mode must be 'z' or 'v'.")


    # --- electrodynamic current profile (segment midpoints) ---
    I_seg_base_A = None
    Em_ref_Vpm = 0.0
    u_current_I = e0.copy()

    tB = np.asarray(Bmeta["t_grid"], dtype=float)
    Em_arr = np.zeros_like(tB, dtype=float)
    for ii, tt in enumerate(tB):
        _, v_tt, _ = circular_orbit_eci_rv(float(tt), sim.alt_m, sim.inc_deg, sim.raan_deg, sim.nu0_deg)
        B_tt = B_eci_of_t(float(tt))
        Em_arr[ii] = float(np.dot(np.cross(v_tt, B_tt), u_current_I))

    Em_ref_Vpm = float(np.mean(np.abs(Em_arr)))
    seg_mid_s_m = (np.arange(1, N_particles + 1, dtype=float) - 0.5) * l0

    try:
        I_seg_base_A, Iprof_meta = compute_bare_tether_current_profile_midpoints(
            L_total_m=L,
            seg_mid_s_m=seg_mid_s_m,
            Em_ref_Vpm=Em_ref_Vpm,
            V_bias_V=float(getattr(sim, "V_bias_V", 0.0)),
            V_cath_V=float(getattr(sim, "V_cath_V", 10.0)),
            plasma_density_m3=float(getattr(sim, "plasma_density_m3", 5e11)),
            tether_diameter_m=float(getattr(sim, "tether_diameter_m", 1e-3)),
            tether_sigma_Spm=float(getattr(sim, "tether_sigma_Spm", 3.5e7)),
            bare_fraction=float(getattr(sim, "bare_fraction", 1.0)),
        )
        print(
            f"  current profile: Em_ref≈{Em_ref_Vpm:.3e} V/m, "
            f"I_mid range≈[{float(np.min(I_seg_base_A)):.3g}, {float(np.max(I_seg_base_A)):.3g}] A",
            flush=True
        )
    except Exception as e:
        print(f"  [warn] current profile failed ({e}); using zero current.", flush=True)
        I_seg_base_A = None


    # initial RELATIVE positions (about COM), RELATIVE velocities
    r0_rel = np.zeros((N_particles, 3), dtype=float)
    v0_rel = np.zeros((N_particles, 3), dtype=float)

    for idx in range(N_particles):
        segnum = idx + 1
        r0_rel[idx, :] = r_attach_rel0 + float(segnum) * l0 * e0

    y0 = np.hstack([r0_rel.reshape(-1), v0_rel.reshape(-1)])

    # sanity: segment lengths at start
    seg_d = np.linalg.norm(np.vstack([r_attach_rel0.reshape(1,3), r0_rel])[1:] - np.vstack([r_attach_rel0.reshape(1,3), r0_rel])[:-1], axis=1)
    print(f"  init sanity: |r_com0|={np.linalg.norm(r_com0):.3e} m, |r_attach_rel0|={np.linalg.norm(r_attach_rel0):.3e} m, "
          f"seg_len min/mean/max={seg_d.min():.3g}/{seg_d.mean():.3g}/{seg_d.max():.3g} m", flush=True)

    # output samples
    t_eval_full = np.linspace(0.0, float(sim.t_final), int(sim.n_output), dtype=float)

    def rhs(t, y):
        return tether_chain_dynamics_rhs(
            t, y, bus, tether, end_mass, sim,
            R_IB_of_t=R_IB_of_t,
            B_eci_of_t=B_eci_of_t,
            I_seg_base_A=I_seg_base_A,
            Em_ref_Vpm=Em_ref_Vpm,
            u_current_I=u_current_I,
        )
    # vector tolerances tuned for RELATIVE state
    atol_vec = np.hstack([
        np.full(3*N_particles, float(sim.atol_pos_m), dtype=float),
        np.full(3*N_particles, float(sim.atol_vel_mps), dtype=float),
    ])

    def rhs_safe(t, y):
        dy = rhs(t, y)
        if not np.all(np.isfinite(dy)):
            raise FloatingPointError(f"non-finite RHS at t={t}")
        return dy
    method_name = getattr(sim, "method", "BDF")
    chunk_s = float(getattr(sim, "chunk_s", 60.0))

    print(f"\n  Starting solve_ivp (stiff; chunked; method={method_name})...", flush=True)
    t2 = time.perf_counter()

    t_cur = 0.0
    y_cur = y0.copy()
    t_final = float(sim.t_final)

    t_chunks = []
    y_chunks = []
    seg_idx = 0

    settled = False
    t_settle = t_final
    hold = float(sim.hold_time)
    vtol = float(sim.v_tol)

    def check_settle_recent(t_arr: np.ndarray, Ymat: np.ndarray) -> Tuple[bool, float]:
        if len(t_arr) == 0 or t_arr[-1] < hold:
            return False, t_final
        t_end = float(t_arr[-1])
        t_start = t_end - hold
        mask = (t_arr >= t_start) & (t_arr <= t_end)
        if not np.any(mask):
            return False, t_final
        Np = N_particles
        Y = Ymat.T
        v_hist = Y[:, 3*Np:6*Np].reshape((-1, Np, 3))
        vmax = float(np.max(np.linalg.norm(v_hist[mask], axis=2)))
        if vmax < vtol:
            return True, float(t_start)
        return False, t_final

    while t_cur < t_final - 1e-12:
        t_next = min(t_cur + chunk_s, t_final)

        mask_eval = (t_eval_full >= t_cur - 1e-12) & (t_eval_full <= t_next + 1e-12)
        t_eval_seg = np.unique(np.concatenate([t_eval_full[mask_eval], np.array([t_cur, t_next])]))
        t_eval_seg = t_eval_seg[(t_eval_seg >= t_cur - 1e-12) & (t_eval_seg <= t_next + 1e-12)]

        print(f"    segment {seg_idx}: integrating [{t_cur:.2f}, {t_next:.2f}] s (samples={len(t_eval_seg)})...", flush=True)

        sol_seg = solve_ivp(
            rhs_safe,
            t_span=(t_cur, t_next),
            y0=y_cur,
            method=method_name,
            t_eval=t_eval_seg,
            rtol=float(sim.rtol),
            atol=atol_vec,
            jac_sparsity=None,
        )
        if not sol_seg.success:
            raise RuntimeError(f"Integration failed in segment {seg_idx}: {sol_seg.message}")

        # stitch (avoid duplicate)
        if seg_idx == 0:
            t_chunks.append(sol_seg.t.copy())
            y_chunks.append(sol_seg.y.copy())
        else:
            t_chunks.append(sol_seg.t[1:].copy())
            y_chunks.append(sol_seg.y[:, 1:].copy())

        t_cur = float(sol_seg.t[-1])
        y_cur = sol_seg.y[:, -1].copy()
        seg_idx += 1

        if bool(getattr(sim, "early_stop", True)):
            t_arr = np.concatenate(t_chunks) if len(t_chunks) else np.array([])
            Ymat = np.hstack(y_chunks) if len(y_chunks) else np.zeros((len(y0), 0))
            ok, t_set = check_settle_recent(t_arr, Ymat)
            if ok:
                settled = True
                t_settle = t_set
                print(f"    settled detected (vmax < {vtol:g} for {hold:g}s) at t≈{t_settle:.2f}s — stopping early.", flush=True)
                break

    t_arr = np.concatenate(t_chunks) if len(t_chunks) else np.array([])
    Ymat = np.hstack(y_chunks) if len(y_chunks) else np.zeros((len(y0), 0))

    t3 = time.perf_counter()
    print(f"  solve_ivp finished in {t3 - t2:.2f} s (segments={seg_idx}, Nt={len(t_arr)})", flush=True)

    # choose output time index
    if settled:
        idx_set = int(np.searchsorted(t_arr, t_settle))
    else:
        idx_set = len(t_arr) - 1
    idx_set = int(min(max(idx_set, 0), len(t_arr) - 1))

    t_out = float(t_arr[idx_set])

    # unpack relative particles at output
    Np = N_particles
    Y = Ymat.T
    r_hist_rel = Y[:, 0:3*Np].reshape((-1, Np, 3))

    # --- Export tether geometry for MATLAB plotting (Option 1) ---
    # Save node positions (including bus attachment point) in ECI over time to a .mat file.
    try:
        r_com_hist = np.zeros((len(t_arr), 3), dtype=float)
        r_attach_hist = np.zeros((len(t_arr), 3), dtype=float)
        for kk, tt in enumerate(t_arr):
            r_com_tt, _, _ = circular_orbit_eci_rv(float(tt), sim.alt_m, sim.inc_deg, sim.raan_deg, sim.nu0_deg)
            R_tt = R_IB_of_t(float(tt))
            r_attach_rel_tt = (R_tt @ tether.attach_B.reshape(3))
            r_com_hist[kk, :] = r_com_tt
            r_attach_hist[kk, :] = r_com_tt + r_attach_rel_tt

        # particles absolute: (Nt, N_particles, 3)
        r_particles_hist_I = r_hist_rel + r_com_hist[:, None, :]

        # full node chain: include attachment as node 0, then particles as nodes 1..N_seg
        r_nodes = np.concatenate([r_attach_hist[:, None, :], r_particles_hist_I], axis=1)

        
        # --- Lorentz force export support (optional for MATLAB arrows) ---
        # Compute COM velocity (finite difference) for Em scaling if enabled
        v_com_hist = np.zeros_like(r_com_hist)
        if len(t_arr) >= 2:
            dt_arr = np.diff(t_arr)
            # central differences for interior, forward/backward for ends
            v_com_hist[0, :] = (r_com_hist[1, :] - r_com_hist[0, :]) / max(t_arr[1]-t_arr[0], 1e-12)
            v_com_hist[-1, :] = (r_com_hist[-1, :] - r_com_hist[-2, :]) / max(t_arr[-1]-t_arr[-2], 1e-12)
            for kk in range(1, len(t_arr)-1):
                dtc = max(t_arr[kk+1] - t_arr[kk-1], 1e-12)
                v_com_hist[kk, :] = (r_com_hist[kk+1, :] - r_com_hist[kk-1, :]) / dtc

        # Segment Lorentz forces history: (Nt, N_seg, 3)
        fL_seg_hist = np.zeros((len(t_arr), int(N_seg), 3), dtype=float)
        I_seg_hist = np.zeros((len(t_arr), int(N_seg)), dtype=float)
        B_hist = np.zeros((len(t_arr), 3), dtype=float)

        for kk, tt in enumerate(t_arr):
            Bk = B_eci_of_t(float(tt))
            B_hist[kk, :] = Bk

            # rebuild per-segment current with same scaling used in RHS
            if I_seg_base_A is None:
                Iseg_k = np.zeros((int(N_seg),), dtype=float)
            else:
                scale = 1.0
                if bool(getattr(sim, 'scale_with_Em', True)) and (u_current_I is not None) and float(Em_ref_Vpm) > 1e-12:
                    Em_k = float(np.dot(np.cross(v_com_hist[kk, :], Bk), u_current_I.reshape(3)))
                    scale *= Em_k / float(Em_ref_Vpm)
                scale *= turn_on_multiplier(float(tt), float(getattr(sim, 'current_turnon_tau_s', 0.0)))
                Iseg_k = np.asarray(I_seg_base_A, dtype=float) * scale

            I_seg_hist[kk, :] = Iseg_k

            # segment vectors in inertial coordinates (between consecutive nodes)
            nodes_k = r_nodes[kk, :, :]  # (N_nodes, 3) with node0=attach, nodeN=end
            for j in range(int(N_seg)):
                dl = nodes_k[j+1, :] - nodes_k[j, :]
                fL_seg_hist[kk, j, :] = float(Iseg_k[j]) * np.cross(dl, Bk)

        # --- end Lorentz force export support ---
export_path = "tether_nodes.mat"
        savemat(export_path, {
            "t": t_arr.astype(float),
            "r_nodes": r_nodes.astype(float),        # (Nt, N_nodes, 3)
            "x_bus": r_com_hist.astype(float),       # (Nt, 3)
            "fL_seg": fL_seg_hist.astype(float),     # (Nt, N_seg, 3) segment Lorentz force vectors
            "I_seg": I_seg_hist.astype(float),       # (Nt, N_seg) segment currents used for forces
            "B_eci": B_hist.astype(float),           # (Nt, 3) magnetic field in ECI used for forces
        })
        print(f"  exported tether node history for MATLAB -> {export_path}  (r_nodes shape={r_nodes.shape})", flush=True)
        except Exception as e:
        print(f"  (note) tether export skipped: {e}", flush=True)
r_out_rel = r_hist_rel[idx_set]

    # bus COM at output
    r_com_out, _, _ = circular_orbit_eci_rv(t_out, sim.alt_m, sim.inc_deg, sim.raan_deg, sim.nu0_deg)
    R_out = R_IB_of_t(t_out)

    # true particle masses (NO mass_scale for inertia)
    m_node_true = float(tether.mass) / float(N_internal) if N_internal > 0 else 0.0
    m_end_true = float(end_mass.mass)
    m_particles = np.full((Np,), m_node_true, dtype=float)
    m_particles[-1] = m_end_true

    # absolute particle positions for inertia
    r_particles_I = r_out_rel + r_com_out

    # inertia tensors
    I_bus_B = box_inertia_about_com(bus.mass, bus.a, bus.b, bus.c)
    I_bus_I = inertia_body_to_inertial(I_bus_B, R_out)
    I_tether_I = compute_inertia_from_particles_about_bus_com_in_inertial(r_particles_I, m_particles, r_com_out)
    I_total_I = I_bus_I + I_tether_I

    return {
        "success": True,
        "settled": settled,
        "t_settle": t_out,
        "r_particles_I": r_particles_I,
        "r_rel_I": r_out_rel,
        "r_com_I": r_com_out,
        "R_IB": R_out,
        "I_bus_I": I_bus_I,
        "I_tether_I": I_tether_I,
        "I_total_I": I_total_I,
        "orbit_period_s": float(Bmeta["period_s"]),
        "Bmeta": Bmeta,
        "solver_method": method_name,
        "mass_scale_used": float(sim.mass_scale),
    }


def read_float(prompt: str) -> float:
    return float(input(prompt).strip())

def read_int(prompt: str) -> int:
    return int(input(prompt).strip())

def read_vec3(prompt: str) -> np.ndarray:
    s = input(prompt).strip()
    parts = [p for p in s.replace(",", " ").split() if p]
    if len(parts) != 3:
        raise ValueError("Enter exactly 3 numbers (x y z).")
    return np.array([float(parts[0]), float(parts[1]), float(parts[2])], dtype=float)

def print_tensor(name: str, I: np.ndarray) -> None:
    np.set_printoptions(precision=6, suppress=True)
    print(f"\n{name} (kg*m^2):\n{I}")


# ============================================================
# Main
# ============================================================

def main() -> None:
    print("=== Satellite MOI Tool (configs 1-3; config 2 is -z_B; config 3 uses WMM+astropy; chunked Radau) ===")
    print("All inertias are about BUS COM.\n")

    # ---------------------------
    # Bus parameters
    # ---------------------------
    print("Enter BUS parameters (uniform rectangular box):")
    bus = BusParams(
        mass=read_float("  Bus mass m_b [kg]: "),
        a=read_float("  Bus size a along x [m]: "),
        b=read_float("  Bus size b along y [m]: "),
        c=read_float("  Bus size c along z [m]: "),
    )

    # ---------------------------
    # (1) Coiled/spool + stowed point mass
    # ---------------------------
    print("\n(1) COILED / STOWED tether as annular cylinder (spool) + stowed point mass:")
    spool = SpoolParams(
        mass=read_float("  Spool (stowed tether) mass m_t(stowed) [kg]: "),
        r_i=read_float("  Spool inner radius r_i [m]: "),
        r_o=read_float("  Spool outer radius r_o [m]: "),
        h=read_float("  Spool thickness/height h [m]: "),
        axis_s_B=read_vec3("  Spool axis direction in BUS frame [sx sy sz]: "),
        com_B=read_vec3("  Spool COM position wrt bus COM in BUS frame [x y z] [m]: "),
    )

    m_p_stowed = read_float("  Stowed end point mass m_p [kg]: ")
    mode = input("  End-mass position mode: [k]=known, [r]=random-in-spool : ").strip().lower()

    if mode == "k":
        r_end_B = read_vec3("  Enter end mass position in BUS frame wrt bus COM [x y z] [m]: ")
    elif mode == "r":
        seed = read_int("  Random seed (int): ")
        rng = np.random.default_rng(seed)
        r_end_B = sample_point_in_annular_cylinder(
            r_i=spool.r_i, r_o=spool.r_o, h=spool.h,
            axis_s_B=spool.axis_s_B, com_B=spool.com_B, rng=rng
        )
        print(f"  Sampled end-mass position in spool: {r_end_B}")
    else:
        raise ValueError("Invalid mode. Choose k or r.")

    I1 = moi_config_1_coiled(bus, spool, m_p_stowed, r_end_B)
    print_tensor("Config (1) MOI (BODY): Bus + Spool + Stowed EndMass", I1)

    # ---------------------------
    # (2) Fully extended tether along -z_B
    # ---------------------------
    print("\n(2) FULLY EXTENDED tether aligned with -z_B (rod + point mass):")
    tether = TetherParams(
        length=read_float("  Tether length L [m]: "),
        mass=read_float("  Tether mass m_t(deployed) [kg]: "),
        attach_B=read_vec3("  Attachment point in BUS frame wrt bus COM [x y z] [m]: "),
    )
    end_mass = PointMassParams(mass=read_float("  End point mass m_p [kg]: "))

    I2, r_p_B0, r_a_B = moi_config_2_extended_on_minus_z(bus, tether, end_mass)
    print_tensor("Config (2) MOI (BODY): Bus + DeployedRod(-z_B) + EndMass", I2)

    print("\n  Geometry vectors (relative to bus COM, in BUS frame at t=0):")
    print(f"  r_a^B0 (attachment) = {r_a_B}")
    print(f"  r_p^B0 (point mass) = {r_p_B0}")

    # ---------------------------
    # (3) Bent tether (orbit + current + WMM2025 + astropy; chunked Radau)
    # ---------------------------
    print("\n(3) BENT TETHER CONFIG (orbit + current + WMM2025 + astropy; stiff):")
    if not ASTROPY_OK:
        print("  [ERROR] astropy not available. Install: pip install astropy")
        return
    if not WMM_OK:
        print("  [ERROR] wmm-calculator not available. Install: pip install wmm-calculator")
        return
    if solve_ivp is None:
        print("  [ERROR] scipy not available. Install: pip install scipy")
        return

    epoch_utc = input("  Epoch UTC: ").strip()

    print("\n  Orbit (circular):")
    alt_m = read_float("  Altitude above Earth [m]: ")
    inc_deg = read_float("  Inclination [deg]: ")
    raan_deg = read_float("  RAAN [deg]: ")
    nu0_deg = read_float("  Initial true anomaly nu0 [deg]: ")

    _, _, Torb = circular_orbit_eci_rv(0.0, alt_m, inc_deg, raan_deg, nu0_deg)
    print(f"  Orbit period ~ {Torb:.3f} s")

    print("\n  Bus attitude during simulation:")
    att_mode = input("  [i]=identity, [e]=constant ZYX euler (deg): ").strip().lower()
    roll_deg = pitch_deg = yaw_deg = 0.0
    if att_mode == "e":
        roll_deg = read_float("  roll about x [deg]: ")
        pitch_deg = read_float("  pitch about y [deg]: ")
        yaw_deg = read_float("  yaw about z [deg]: ")

    print("\n  Current model:")
    print("  Bare tether current-profile inputs:")
    V_bias_V = read_float("  Bias / supply voltage V_bias [V]: ")
    V_cath_V = read_float("  Cathode drop V_cath [V]: ")
    plasma_density_m3 = read_float("  Plasma density N_inf [1/m^3]: ")
    tether_diameter_m = read_float("  Tether diameter d [m]: ")
    tether_sigma_Spm = read_float("  Conductivity sigma [S/m]: ")
    bare_fraction = read_float("  Bare fraction [0..1]: ")
    scale_with_Em = input("  Scale current with Em(t) along orbit? [y/n]: ").strip().lower() == "y"
    current_turnon_tau_s = read_float("  Current turn-on time constant tau [s]: ")
    print("\n  Tether dynamics discretization:")
    N_seg = read_int("  Number of segments N_seg: ")
    EA_N = read_float("  Axial stiffness EA [N]: ")
    c_Ns_m = read_float("  Axial damping c [N*s/m]: ")
    tension_only = input("  Tension-only (no compression) [y/n]: ").strip().lower() == "y"

    print("\n  Time settings:")
    t_final = read_float("  t_final [s]: ")
    n_output = read_int("  n_output samples: ")

    print("\n  WMM precompute sampling:")
    B_samples = read_int("  B_samples: ")

    print("\n  Settle detection (relative to bus COM translation):")
    v_tol = read_float("  v_tol [m/s]: ")
    hold_time = read_float("  hold_time [s]: ")

    print("\n  Initial tether direction in inertial:")
    init_mode = input("  [z]=use -z_I, [v]=enter vector: ").strip().lower()
    init_dir = None
    if init_mode == "v":
        init_dir = read_vec3("  Enter inertial direction [x y z]: ")
        init_dir = unit(init_dir)

    sim = BentTetherSimParams(
        epoch_utc=epoch_utc,
        alt_m=alt_m, inc_deg=inc_deg, raan_deg=raan_deg, nu0_deg=nu0_deg,
        att_mode=att_mode, roll_deg=roll_deg, pitch_deg=pitch_deg, yaw_deg=yaw_deg,
        V_bias_V=V_bias_V, V_cath_V=V_cath_V,
        plasma_density_m3=plasma_density_m3,
        tether_diameter_m=tether_diameter_m,
        tether_sigma_Spm=tether_sigma_Spm,
        bare_fraction=bare_fraction,
        scale_with_Em=scale_with_Em,
        current_turnon_tau_s=current_turnon_tau_s,
        N_seg=N_seg, EA_N=EA_N, c_Ns_m=c_Ns_m, tension_only=tension_only,
        t_final=t_final, n_output=n_output,
        v_tol=v_tol, hold_time=hold_time,
        init_dir_mode=init_mode, init_dir_I=init_dir,
        B_samples=B_samples
    )

    print("\n  Running bent-tether simulation ...", flush=True)
    results = run_bent_tether_sim(bus, tether, end_mass, sim)

    print(f"\n  Settle status: {'settled' if results['settled'] else 'NOT settled (using t_final)'}")
    print(f"  t_out = {results['t_settle']:.6f} s")
    print(f"  Bus COM (ECI) at t_out: {results['r_com_I']} [m]")
    print(f"  Solver method: {results['solver_method']}")

    print_tensor("Config (3) I_bus (INERTIAL)", results["I_bus_I"])
    print_tensor("Config (3) I_tether+end (INERTIAL)", results["I_tether_I"])
    print_tensor("Config (3) I_total (INERTIAL) about bus COM", results["I_total_I"])

    print("\nDone.")


if __name__ == "__main__":
    main()


SyntaxError: expected 'except' or 'finally' block (3992065453.py, line 1217)